# Modeling and Evaluation

This notebook captures the modeling workflow, validation metrics, and comparison of candidate regressors.

In [ ]:
# Save final model
MODELS_DIR.mkdir(exist_ok=True, parents=True)
joblib.dump(rf, MODELS_DIR / 'final_model.joblib')

# Save model summary
summary = {
    'validation_rmse': float(rf_rmse),
    'validation_mae': float(rf_mae),
    'validation_r2': float(rf_r2),
    'model_type': 'RandomForestRegressor',
    'n_estimators': 150,
    'max_depth': 12,
    'n_features': len(feature_cols),
}

REPORTS_DIR.mkdir(exist_ok=True, parents=True)
with open(REPORTS_DIR / 'D_model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Final model saved to models/final_model.joblib")
print(f"✓ Model summary saved to reports/D_model_summary.json")

In [ ]:
print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

# Baseline: mean predictor
baseline_pred = np.full_like(y_val, y_train.mean())
baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_pred))
baseline_mae = mean_absolute_error(y_val, baseline_pred)
baseline_r2 = r2_score(y_val, baseline_pred)

print(f"\n1. Mean Baseline:")
print(f"   RMSE: {baseline_rmse:.4f}")
print(f"   MAE: {baseline_mae:.4f}")
print(f"   R²: {baseline_r2:.4f}")

# Linear regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_val)
lr_rmse = np.sqrt(mean_squared_error(y_val, lr_pred))
lr_mae = mean_absolute_error(y_val, lr_pred)
lr_r2 = r2_score(y_val, lr_pred)

print(f"\n2. Linear Regression:")
print(f"   RMSE: {lr_rmse:.4f}")
print(f"   MAE: {lr_mae:.4f}")
print(f"   R²: {lr_r2:.4f}")

# Random Forest
rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_val)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))
rf_mae = mean_absolute_error(y_val, rf_pred)
rf_r2 = r2_score(y_val, rf_pred)

print(f"\n3. Random Forest:")
print(f"   RMSE: {rf_rmse:.4f}")
print(f"   MAE: {rf_mae:.4f}")
print(f"   R²: {rf_r2:.4f}")

# Save comparison
comparison_df = pd.DataFrame({
    'model': ['mean_baseline', 'linear_regression', 'random_forest'],
    'rmse': [baseline_rmse, lr_rmse, rf_rmse],
    'mae': [baseline_mae, lr_mae, rf_mae],
    'r2': [baseline_r2, lr_r2, rf_r2],
})
comparison_df.to_csv(REPORTS_DIR / 'D_model_comparison.csv', index=False)
print(f"\n✓ Model comparison saved to reports/D_model_comparison.csv")

In [ ]:
# Prepare data
feature_cols = [c for c in train.columns if c not in ['plot_id', 'yield_tons_per_ha', 'price_birr_per_quintal', 'revenue_per_ha_birr']]
X = train[feature_cols].fillna(0)
y = train['yield_tons_per_ha']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Feature columns: {len(feature_cols)}")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

import pandas as pd
import numpy as np
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
REPORTS_DIR = Path('../reports')

# Load data
train = pd.read_csv(PROCESSED_DIR / 'master_train.csv')
test = pd.read_csv(PROCESSED_DIR / 'master_test.csv')

print(f"Training data: {train.shape}")
print(f"Test data: {test.shape}")
print(f"Target: yield_tons_per_ha")